In [1]:
import kagglehub
import pandas as pd
path = kagglehub.dataset_download("ealaxi/banksim1")
print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/banksim1


In [25]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve
from sklearn.preprocessing import StandardScaler
import joblib
import json

def load_and_preprocess(filepath):
    df = pd.read_csv(filepath, quotechar='"')

    str_cols = ['customer', 'merchant', 'category', 'gender', 'zipcodeOri', 'zipMerchant']
    for col in str_cols:
        df[col] = df[col].str.replace("'", "")

    df['amount_log'] = np.log(df['amount'] + 1e-6)
    df['hour'] = df['step'] % 24

    df = pd.get_dummies(df, columns=['category', 'gender', 'age'])

    return df

def train_xgboost_model(X_train, y_train):
    scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])

    params = {
        'objective': 'binary:logistic',
        'eval_metric': ['logloss', 'aucpr'],
        'scale_pos_weight': scale_pos_weight,
        'max_depth': 5,
        'learning_rate': 0.1,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'gamma': 1,
        'min_child_weight': 10,
        'seed': 42
    }

    dtrain = xgb.DMatrix(X_train, label=y_train)
    model = xgb.train(
        params,
        dtrain,
        num_boost_round=500,
        early_stopping_rounds=50,
        evals=[(dtrain, 'train')],
        verbose_eval=20
    )

    return model

def find_optimal_threshold(model, X_test, y_test):
    dtest = xgb.DMatrix(X_test)
    y_proba = model.predict(dtest)

    precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-9)
    optimal_idx = np.argmax(f1_scores)
    optimal_threshold = thresholds[optimal_idx]

    return optimal_threshold

def save_model_pipeline(model, features, threshold, file_prefix='fraud_model'):
    model.save_model(f'{file_prefix}.json')

    pipeline = {
        'model': model,
        'features': features,
        'threshold': threshold
    }
    joblib.dump(pipeline, f'{file_prefix}.pkl')

    metadata = {
        'features': features,
        'threshold': float(threshold)
    }
    with open(f'{file_prefix}_metadata.json', 'w') as f:
        json.dump(metadata, f)

if __name__ == '__main__':
    df = load_and_preprocess(path+'/bs140513_032310.csv')

    features = ['amount', 'amount_log', 'hour'] + \
              [col for col in df.columns if col.startswith(('category_', 'gender_', 'age_'))]
    X = df[features]
    y = df['fraud']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train[['amount', 'amount_log', 'hour']] = scaler.fit_transform(X_train[['amount', 'amount_log', 'hour']])
    X_test[['amount', 'amount_log', 'hour']] = scaler.transform(X_test[['amount', 'amount_log', 'hour']])

    print("Training XGBoost model...")
    model = train_xgboost_model(X_train, y_train)

    print("\nFinding optimal threshold...")
    threshold = find_optimal_threshold(model, X_test, y_test)
    print(f"Optimal threshold: {threshold:.4f}")

    dtest = xgb.DMatrix(X_test)
    y_proba = model.predict(dtest)
    y_pred = (y_proba >= threshold).astype(int)

    print("\nFinal Model Performance:")
    print(classification_report(y_test, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nSaving model pipeline...")
    save_model_pipeline(model, features, threshold)
    joblib.dump(scaler, 'scaler.pkl')
    print("Model saved as fraud_model.pkl, fraud_model.json, and scaler.pkl")

Training XGBoost model...
[0]	train-logloss:0.61071	train-aucpr:0.70044
[20]	train-logloss:0.15972	train-aucpr:0.77831
[40]	train-logloss:0.11901	train-aucpr:0.80411
[60]	train-logloss:0.11161	train-aucpr:0.81456
[80]	train-logloss:0.10922	train-aucpr:0.81806
[100]	train-logloss:0.10647	train-aucpr:0.82164
[120]	train-logloss:0.10339	train-aucpr:0.82462
[140]	train-logloss:0.09996	train-aucpr:0.82847
[160]	train-logloss:0.09779	train-aucpr:0.83039
[180]	train-logloss:0.09516	train-aucpr:0.83258
[200]	train-logloss:0.09336	train-aucpr:0.83502
[220]	train-logloss:0.09161	train-aucpr:0.83735
[240]	train-logloss:0.08947	train-aucpr:0.83952
[260]	train-logloss:0.08742	train-aucpr:0.84152
[280]	train-logloss:0.08598	train-aucpr:0.84312
[300]	train-logloss:0.08486	train-aucpr:0.84479
[320]	train-logloss:0.08357	train-aucpr:0.84631
[340]	train-logloss:0.08209	train-aucpr:0.84789
[360]	train-logloss:0.08088	train-aucpr:0.84939
[380]	train-logloss:0.07956	train-aucpr:0.85078
[400]	train-logloss:

In [29]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib

new_data = pd.DataFrame({
    'step': [345, 678, 901],
    'customer': ['C12345', 'C67890', 'C13579'],
    'age': ['3', '2', '1'],
    'gender': ['M', 'F', 'M'],
    'category': ['es_barsandrestaurants', 'es_health', 'es_transportation'],
    'amount': [85.20, 150.00, 32.50]
})

new_data['amount_log'] = np.log(new_data['amount'] + 1e-6)
new_data['hour'] = new_data['step'] % 24

categories = [
    'es_barsandrestaurants', 'es_health', 'es_transportation', 'es_food',
    'es_contents', 'es_fashion', 'es_home', 'es_hotelservices', 'es_hyper',
    'es_leisure', 'es_otherservices', 'es_sportsandtoys', 'es_tech',
    'es_travel', 'es_wellnessandbeauty'
]
genders = ['M', 'F', 'E', 'U']
ages = ['0', '1', '2', '3', '4', '5', '6', 'U']

for cat in categories:
    new_data[f'category_{cat}'] = (new_data['category'] == cat).astype(int)
for gender in genders:
    new_data[f'gender_{gender}'] = (new_data['gender'] == gender).astype(int)
for age in ages:
    new_data[f'age_{age}'] = (new_data['age'] == age).astype(int)

pipeline = joblib.load('fraud_model.pkl')
scaler = joblib.load('scaler.pkl')

features_to_use = pipeline['features']

new_data_full = pd.DataFrame(0, index=new_data.index, columns=features_to_use)
for col in new_data.columns:
    if col in features_to_use:
        new_data_full[col] = new_data[col]

num_features = ['amount', 'amount_log', 'hour']
new_data_full[num_features] = scaler.transform(new_data_full[num_features])

dmatrix = xgb.DMatrix(new_data_full[features_to_use])
probas = pipeline['model'].predict(dmatrix)
predictions = (probas >= pipeline['threshold']).astype(int)

new_data['fraud_probability'] = probas
new_data['fraud_prediction'] = predictions

print("Prediction Results:")
print(new_data[['customer', 'amount', 'category', 'fraud_probability', 'fraud_prediction']])

Prediction Results:
  customer  amount               category  fraud_probability  fraud_prediction
0   C12345    85.2  es_barsandrestaurants           0.297623                 0
1   C67890   150.0              es_health           0.815823                 0
2   C13579    32.5      es_transportation           0.000024                 0
